# Choosing k Without Fooling Yourself

**Topic:** k-NN regression &nbsp;·&nbsp; **Difficulty:** Medium

`k = 1` gives zero training error and terrible test error: every point is its
own nearest neighbour, so training error measures nothing at all. This is the
bias-variance trade-off in its purest visible form, and k-NN is where it is
easiest to see.

Sweep `k`, score honestly with cross-validation, and find where the U-shaped
curve bottoms out.

## The data (created in the next cell)

- `X` `(500, 4)`, `y` `(500,)`: a smooth nonlinear function plus noise.
- `ks = [1, 2, 3, 5, 8, 12, 20, 35, 60, 100]`.

## Your task

With **NumPy only** for the k-NN averaging itself (`sklearn.model_selection.KFold` is allowed for the fold split):

1. Standardize `X` once using the full data (acceptable here, since it carries no
   target information).
2. For each `k`, compute:
   - `train_rmse`: predicting each training point from its own `k` nearest
     neighbours **including itself**,
   - `cv_rmse`: 5-fold cross-validated RMSE with
     `KFold(n_splits=5, shuffle=True, random_state=0)`, predicting each held-out
     point as the mean of its `k` nearest **training** neighbours.
3. Set `answers` to a dict with `"train_rmse"`, `"cv_rmse"` (arrays over `ks`),
   `"best_k"` (the `k` minimizing CV RMSE), and `"train_rmse_k1"`.

## Evaluation

Compared to the reference to 1e-6. The evaluation cell checks that training RMSE
at `k = 1` is essentially zero while CV RMSE there is not.

In [ ]:
import numpy as np

rng = np.random.default_rng(808)
n = 500
X = rng.uniform(-2.5, 2.5, (n, 4))
y = (np.sin(1.4 * X[:, 0]) + 0.6 * X[:, 1] ** 2 - 0.5 * X[:, 2] * X[:, 3]
     + rng.normal(0, 0.25, n))
ks = [1, 2, 3, 5, 8, 12, 20, 35, 60, 100]
print(f"X {X.shape}, y std = {y.std():.3f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("train_rmse", [10], 7.001940084284689, 0.32288181346587413, 1e-06),
    ("cv_rmse", [10], 9.347811669309083, 0.3534948902001134, 1e-06),
    ("best_k", [], 5.0, -0.8371900740724958, 1e-06),
    ("train_rmse_k1", [], 0.0, 0.0, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "train_rmse_k1" in answers and "cv_rmse" in answers:
    _z = answers["train_rmse_k1"] < 1e-12 and np.asarray(answers["cv_rmse"])[0] > 0.1
    print(f"  {'k=1 train error is 0':<24} {'ok' if _z else 'wrong'}")
    _ok = _ok and _z

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

At `k = 1` the training prediction is the point's own value, so training RMSE is
exactly zero and tells you nothing, which is why "it fits the training data
perfectly" is never evidence of anything. Cross-validation shows the real curve:
error falls as averaging suppresses noise, then rises as the neighbourhood grows
large enough to smooth away the signal.

```python
import numpy as np
from sklearn.model_selection import KFold

Z = (X - X.mean(axis=0)) / X.std(axis=0)
n = len(y)

D_all = np.linalg.norm(Z[:, None, :] - Z[None, :, :], axis=2)
order_all = np.argsort(D_all, axis=1)

train_rmse, cv_rmse = [], []
kf = KFold(n_splits=5, shuffle=True, random_state=0)
folds = list(kf.split(Z))

for k in ks:
    nn = order_all[:, :k]
    pred = y[nn].mean(axis=1)
    train_rmse.append(float(np.sqrt(np.mean((pred - y) ** 2))))

    sq, cnt = 0.0, 0
    for tr, te in folds:
        D = np.linalg.norm(Z[te][:, None, :] - Z[tr][None, :, :], axis=2)
        idx = np.argsort(D, axis=1)[:, :k]
        p = y[tr][idx].mean(axis=1)
        sq += float(((p - y[te]) ** 2).sum())
        cnt += len(te)
    cv_rmse.append(float(np.sqrt(sq / cnt)))

train_rmse = np.array(train_rmse)
cv_rmse = np.array(cv_rmse)

answers = {
    "train_rmse": train_rmse,
    "cv_rmse": cv_rmse,
    "best_k": int(ks[int(np.argmin(cv_rmse))]),
    "train_rmse_k1": float(train_rmse[0]),
}
```